# AI Hiring Fairness Auditor — Exploration
Use this notebook to explore the dataset, visualise bias, and test ideas before moving to production code.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.data_loader import load_as_dataframe

X_train, X_test, y_train, y_test, s_train, s_test = load_as_dataframe()
print('Data loaded.')

## 1. Dataset overview

In [ ]:
print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print()
print('Shortlist rate (overall):', y_test.mean().round(3))
print('Shortlist rate (male)   :', y_test[s_test == 1].mean().round(3))
print('Shortlist rate (female) :', y_test[s_test == 0].mean().round(3))

## 2. Shortlisting rate by gender (raw data)

In [ ]:
rates = {
    'Male':   y_test[s_test == 1].mean(),
    'Female': y_test[s_test == 0].mean()
}
plt.bar(rates.keys(), rates.values(), color=['#378ADD', '#D4537E'], width=0.4)
plt.axhline(0.8 * rates['Male'], color='red', linestyle='--', label='80% rule threshold')
plt.ylabel('Shortlist rate')
plt.title('Shortlisting rate by gender — raw data')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Disparate Impact Ratio: {rates['Female'] / rates['Male']:.3f}  (threshold: 0.8)")

## 3. Train baseline model and check fairness

In [ ]:
from src.model import train_baseline
from src.bias_detector import compute_fairness_metrics, print_fairness_report

model = train_baseline(X_train, y_train)
y_pred = model.predict(X_test)

metrics = compute_fairness_metrics(y_test.values, y_pred, s_test.values)
print_fairness_report(metrics, 'Baseline model')

## 4. Apply reweighing and compare

In [ ]:
from src.bias_mitigator import train_reweighed

debiased = train_reweighed(X_train, y_train, s_train)
y_pred_fair = debiased.predict(X_test)
metrics_fair = compute_fairness_metrics(y_test.values, y_pred_fair, s_test.values)
print_fairness_report(metrics_fair, 'After reweighing')

# Side-by-side bar chart
labels  = ['Dem. Parity Diff', 'Disparate Impact\n(distance from 1)', 'Equal Opp. Diff']
before  = [abs(metrics['demographic_parity_diff']),
           abs(1 - metrics['disparate_impact_ratio']),
           abs(metrics['equal_opportunity_diff'])]
after   = [abs(metrics_fair['demographic_parity_diff']),
           abs(1 - metrics_fair['disparate_impact_ratio']),
           abs(metrics_fair['equal_opportunity_diff'])]

x = np.arange(len(labels))
fig, ax = plt.subplots()
ax.bar(x - 0.2, before, 0.4, label='Baseline', color='#E24B4A', alpha=0.85)
ax.bar(x + 0.2, after,  0.4, label='Debiased', color='#1D9E75', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Bias magnitude  (lower = fairer)')
ax.set_title('Bias before vs after mitigation')
ax.legend()
plt.tight_layout()
plt.show()

## 5. SHAP — explain a single decision

In [ ]:
import shap
from src.explainer import get_shap_explainer, explain_single, top_factors

explainer, scaler = get_shap_explainer(model, X_train)

# Pick a rejected female candidate
sample_idx = X_test[(s_test.values == 0) & (y_pred == 0)].index[0]
sample = X_test.loc[[sample_idx]]

shap_dict = explain_single(explainer, scaler, sample)
factors = top_factors(shap_dict, n=8)

feats  = [f for f, _ in factors]
vals   = [v for _, v in factors]
colors = ['#1D9E75' if v > 0 else '#E24B4A' for v in vals]

plt.barh(feats[::-1], vals[::-1], color=colors[::-1])
plt.axvline(0, color='gray', linewidth=0.8)
plt.xlabel('SHAP value  (positive = supports shortlisting)')
plt.title('Decision explanation — rejected female candidate')
plt.tight_layout()
plt.show()